In [22]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import ClientSecretCredential
# Import MCPTool from the agents/tools module
from azure.ai.projects.models import MCPTool

In [ ]:
tenant_id = " "
client_id = " "
client_secreate = " "
end_point = "9"
model_name = "gpt-5-mini"
mcp_connection_name = "azure-doc-mcp"

In [24]:
mcp_url = "https://learn.microsoft.com/api/mcp"

In [25]:
credentials = ClientSecretCredential(
    client_id=client_id,
    tenant_id=tenant_id,
    client_secret=client_secreate
)

In [26]:
project_client = AIProjectClient(
    endpoint=end_point, 
    client_id=client_id, 
    credential=credentials
)

In [31]:
mcp_tool = MCPTool(
    server_label="my-mcp-server",
    server_url=mcp_url,
    require_approval="always"  # Options: "always", "never", etc.
)

In [32]:
openai_client = project_client.get_openai_client()

In [33]:
response = openai_client.responses.create(
    model=model_name, 
    input="List the tools available from the Microsoft Learn MCP server. Explain what each tool does..",
    tools=[mcp_tool] 
)

In [34]:
print(response.output_text)

In [35]:
response

Response(id='resp_0de91b1eef8bd2a7006ad39ee5bd388195884a94c466875ac8', created_at=1792253669.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini', object='response', output=[McpListTools(id='mcpl_0de91b1eef8bd2a7006ad39ee5d8248195b7f81bbc19608629', server_label='my-mcp-server', tools=[McpListToolsTool(input_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'a query or topic about Microsoft/Azure products, services, platforms, developer tools, frameworks, or APIs', 'properties': {}, 'default': None, 'required': []}}, 'required': []}, name='microsoft_docs_search', annotations={}, description="Search official Microsoft/Azure documentation to find the most relevant and trustworthy content for a user's query. This tool returns up to 10 high-quality content chunks (each max 500 tokens), extracted from Microsoft Learn and other official sources. Each result includes the article title, URL, and a self-contained content exc

In [36]:
print("--- Response Output Items ---")
for item in response.output:
    # Check if the output item is an MCP approval request
    if item.type == "mcp_approval_request":
        print(f"\n[APPROVAL REQUIRED]")
        print(f"Tool Name: {item.name}")
        print(f"Server Label: {item.server_label}")
        print(f"Arguments: {item.arguments}")
        print(f"Approval Request ID: {item.id}")
    elif item.type == "mcp_list_tools":
        print(f"\n[MCP Tools Discovered from {item.server_label}]:")
        for t in item.tools:
            print(f" - {t.name}: {t.description[:80]}...")

--- Response Output Items ---

[MCP Tools Discovered from my-mcp-server]:
 - microsoft_docs_search: Search official Microsoft/Azure documentation to find the most relevant and trus...
 - microsoft_code_sample_search: Search for code snippets and examples in official Microsoft Learn documentation....
 - microsoft_docs_fetch: Fetch and convert a Microsoft Learn documentation webpage to markdown format. Th...

[APPROVAL REQUIRED]
Tool Name: microsoft_docs_search
Server Label: my-mcp-server
Arguments: {"query":"MCP server tools Microsoft Learn list 'MCP server' 'tools' 'Microsoft Learn'"}
Approval Request ID: mcpr_0de91b1eef8bd2a7006ad39ee7ad74819597e6726ed7f02456


In [37]:
approval_handled = False

for item in response.output:
    if item.type == "mcp_approval_request":
        print("\n" + "="*40)
        print("[HUMAN-IN-THE-LOOP APPROVAL REQUIRED]")
        print(f"Tool Name: {item.name}")
        print(f"Server: {item.server_label}")
        print(f"Arguments: {item.arguments}")
        print("="*40)
        
        # Pause execution and wait for human input
        choice = input("Do you want to approve this tool execution? (y/n): ").strip().lower()
        is_approved = (choice == 'y')
        
        print(f"Sending approval decision (Approved: {is_approved}) back to model...")
        
        # 5. Resume execution: MUST pass tools=[mcp_tool] here so the server remains enabled
        response = openai_client.responses.create(
            model=model_name,
            previous_response_id=response.id,
            tools=[mcp_tool],  # <--- CRITICAL: Keeps the MCP server enabled during approval
            input=[
                {
                    "type": "mcp_approval_response",
                    "approve": is_approved,
                    "approval_request_id": item.id
                }
            ]
        )
        approval_handled = True
        break

# 6. Print final results
if approval_handled:
    print("\n--- Final Execution Output ---")
    print(response.output_text)
else:
    print("\n--- Direct Output ---")
    print(response.output_text)


[HUMAN-IN-THE-LOOP APPROVAL REQUIRED]
Tool Name: microsoft_docs_search
Server: my-mcp-server
Arguments: {"query":"MCP server tools Microsoft Learn list 'MCP server' 'tools' 'Microsoft Learn'"}
Sending approval decision (Approved: True) back to model...

--- Final Execution Output ---
Short answer — the Learn MCP Server currently exposes three tools:

1) microsoft_docs_search  
- What it does: performs semantic search over Microsoft Learn documentation. Returns relevant articles/snippets and links so an agent can find where a concept or instruction is covered.

2) microsoft_docs_fetch  
- What it does: fetches a full Learn documentation page and converts it to markdown (full article content). Useful for deep research or multi-turn answers that need the complete source text.

3) microsoft_code_sample_search  
- What it does: searches Microsoft Learn for code samples and examples, returning relevant snippets and references (API examples, sample code) for developer queries.

Notes:
- Thes